<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/interp_engine_example/interp_engine_hub_adapter.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

In [ ]:
# Uncomment to run installation steps if you do not have a development
# editable install and want to run this notebook in a fresh environment.
# %pip install uv
# %uv pip install --upgrade pip setuptools wheel && \
# %uv pip install 'git+https://github.com/speediedan/interpretune.git@main[examples]'
# %uv pip install --group git-deps
#
# NOTE: This cell is intentionally commented out. We will uncomment these
# install commands once we no longer need to preserve editable installs
# for active developer venvs.


# A hub-delivered adapter, and the hook name that lies

This is the notebook form of `it_examples.experiments.interp_engine.hub_adapter_demo`. It shows the
same three things, on GPU:

1. **A third-party adapter arriving through the Hub.** The adapter is not bundled in interpretune. It is
   pulled as a component, gated by the trust opt-in, and registers `Adapter.interp_engine` into the
   composition registry on load. Nothing in interpretune knows about interp-engine at build time.
2. **Capture at the tensor a transcoder is actually trained on**, through the adapter's seam.
3. **The hook-name hazard, made concrete.** Asking for the same thing under TransformerLens' legacy
   block-level name is REFUSED with an explanation rather than silently answered with a neighbouring
   tensor.

Point 3 is the one worth your attention. Gemma Scope 2 transcoders declare
`pre_feedforward_layernorm.output` as their input; SAELens declares the TransformerLens name
`blocks.{i}.hook_mlp_in` for the same artifacts; and TransformerLens fires that hook on the residual
stream **before** the norm. On this model at layer 5 the two tensors share a cosine similarity of
0.088 — enough for a dashboard to read as ordinary while encoding activations its transcoder was never
trained on.

## Why this notebook awaits, and the CLI script does not

interp-engine's synchronous free functions refuse to run inside a live event loop, and Jupyter always
has one. That refusal is a feature: it makes the mismatch visible rather than deadlocking. So the
notebook uses the engine's **async** surface, reached through the adapter's seam, while the CLI script
uses the sync one because a script has no loop.

We deliberately do not reach for `nest_asyncio`. Patching the running loop to force a sync API through
hides the reason that API refused, and the engine offers a real async path.

In [ ]:
# Parameters - injected by papermill during parameterized runs.
component = "speediedan/it-interp-engine-adapter"
# Pin the revision rather than tracking a branch: a component is remote code, and a pinned revision
# cannot change under you between the read and the run.
revision = None  # set to a published revision sha
model_name = "google/gemma-3-1b-it"
layer = 5
text = "The capital of France is Paris, and the capital of Germany is Berlin."

## 1. Pull the component, and opt in deliberately

The trust gate refuses hub-resident code by default. Opting in is an act, not boilerplate: the line
below is part of the demonstration. `pull` caches the repo **without executing anything**, so the
entrypoint can be read before it is trusted.

In [ ]:
import os

os.environ.setdefault("IT_TRUST_REMOTE_CODE", "1")

from interpretune.hub import pull
from interpretune.hub.adapters import load_hub_adapter, loaded_adapter_module
from interpretune.protocol import Adapter

before = set(Adapter.__members__)
pull(component, revision=revision)
members = load_hub_adapter(component)
added = sorted(set(Adapter.__members__) - before)
print(f"registered adapters: {[m.name for m in members]}   (new Adapter members: {added})")

`Adapter` is a closed enum in interpretune. A hub component extended it at runtime, and the
composition registry now carries entries the framework did not ship.

## 2. Load the model, and wrap it in place

`EagerModel` takes a model interpretune already holds: no reload and no second copy on the GPU.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

hf_model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name)

seam = loaded_adapter_module(component)._Seam
engine_model = seam.wrap_model(hf_model, tokenizer, None, model_name)
tokens = engine_model.to_tokens(text)
print(f"wrapped {model_name}; {tokens.shape[-1]} tokens")

## 3. Capture the tensor the transcoder was trained on

`blocks.5.mlp.hook_in` names the MLP's argument, which is the block norm's **output**. The seam
translates it to an engine address, and the translation is model-aware: string-only translation
silently differs on sandwich-norm architectures like this one.

In [ ]:
correct = f"blocks.{layer}.mlp.hook_in"
point = seam.point_for(correct, engine_model)

# await, not call: see the note at the top.
captured = (await seam.capture_async(engine_model, tokens, [point]))[point]
print(f"captured {correct} -> {point}   shape {tuple(captured.shape)}")

### Ground truth, from the module the artifact's own config names

A Gemma Scope 2 transcoder's `config.json` names `pre_feedforward_layernorm.output`. Read it directly
with a forward hook and compare.

In [ ]:
truth = {}


def _record(_module, _inputs, output):
    truth["value"] = output.detach()


handle = hf_model.model.layers[layer].pre_feedforward_layernorm.register_forward_hook(_record)
try:
    with torch.no_grad():
        hf_model(tokens)
finally:
    handle.remove()

cos = F.cosine_similarity(captured.flatten().float(), truth["value"].flatten().float(), dim=0)
print(f"cos(seam capture, pre_feedforward_layernorm.output) = {cos.item():.6f}")

## 4. The refusal

Now ask for the same thing under the legacy block-level name. SAELens publishes exactly this name for
these artifacts, so it is the name a reader is most likely to reach for. The adapter refuses rather
than answering with the neighbouring tensor.

In [ ]:
legacy = f"blocks.{layer}.hook_mlp_in"

try:
    seam.point_for(legacy, engine_model)
except Exception as refusal:
    print(f"{type(refusal).__name__}: {refusal}")

Read the message rather than skimming it. The adapter declines to choose for you, and names what to
ask for instead. Substituting silently is what SAELens, interp-engine and interpretune's own mapping
table each did at some point; this is the one place that refuses to be the fourth.

## What to take away

- A component from a repo interpretune does not control extended a closed enum, registered
  compositions, and captured activations — through published rails, behind a trust gate.
- The capture is at the tensor the artifact's own config names, not the one its published hook name
  suggests.
- The two differ by a whole normalization, and nothing about a wrong answer here looks wrong.